# Data-Level Security Using Dynamic Views

This notebook demonstrates how data-level security can be implemented using dynamic views.

The implementation covers two key capabilities:

- **Row-Level Security:** Controls which rows are visible to a user  
- **Column-Level Masking:** Controls how sensitive values are displayed

A simple sales dataset is used to illustrate how different users receive different results when querying the same view.

### Step 1: Set up the schema for the table

In [0]:
%sql
USE CATALOG demo;
CREATE SCHEMA IF NOT EXISTS data_security;

In [0]:
%sql
USE SCHEMA data_security;

### Step 2: Create the table 
A table is defined to represent sales data across multiple regions.

In [0]:
%sql
CREATE OR REPLACE TABLE sales_dynamic (
    id INT,
    region STRING,
    email STRING,
    revune INT
);


###Step 3: Insert Sample data
The dataset includes records from both UK and US regions.
<br>
This enables validation of row-level security by ensuring that different users see only relevant regional data.

In [0]:
%sql
INSERT INTO sales_dynamic VALUES
    (1, 'UK', 'john.doe@email.com', 10000),
    (2, 'US', 'jane.doe@email.com', 20000),
    (3, 'UK', 'jack.smith@email.com', 30000),
    (4, 'US', 'jill.smith@email.com', 40000),
    (5, 'UK', 'jimmy.smith@email.com', 50000),


In [0]:
%sql
Select * FROM sales_dynamic

## Step 4: Create the dynamic view

The dynamic view implements both row-level security and column-level masking.

### Row-Level Security

- **UK group** → returns only rows where `region = 'UK'`
- **US group** → returns only rows where `region = 'US'`
- **Admin group** → returns all rows

### Column-Level Masking

- **Admin group** → full email visible
- **Other users** → email is masked

### Key Function: `is_account_group_member()`

This function evaluates whether the current user belongs to a specified group.

**Examples:**

- `is_account_group_member('uk-sg')`
- `is_account_group_member('us-sg')`
- `is_account_group_member('admin-sg')`

The function returns **TRUE** or **FALSE** and is used to drive conditional logic inside the view.

In [0]:
%sql
SELECT is_account_group_member('admin-sg'), is_account_group_member('uk-sg')

In [0]:
%sql
CREATE OR REPLACE VIEW vw_sales_dynamic 
AS 
SELECT
    id,
    region,
    CASE 
        WHEN is_account_group_member('admin-sg') THEN email
        ELSE concat(substr(email, 1,1), '***@', split(email, '@') [1])
    END AS email,
    revenue
from sales_dynamic
WHERE is_account_group_member('admin-sg')
OR (is_account_group_member('uk-sg') AND region = 'UK')
OR (is_account_group_member('us-sg') AND region = 'US');



### Step 5: Query the dynamic view

In [0]:
%sql
SELECT * FROM vw_sales_dynamic; 